In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import csv
import os
import re
import sys
from collections import Counter

MODERN_FOLDER = "textos-modernizados"                         # From 6-modernizacion, changed here in place
CORRECTIONS_FILE = "ocr-piloto/correcciones-puntuales.csv"    # libro, incorrecta, correcta, tipo
LEXICAL_FILE = "ocr-piloto/modernizacion-lexica.csv"          # antigua, moderna (whole corpus)
FORMS_FILE = "ocr-piloto/modernizacion-formas.csv"            # antigua, moderna, tipo (whole corpus)
REGISTER_FILE = "registro-correcciones.csv"                   # Every change: book, line, before, after, list

sys.path.insert(0, "ocr-piloto")
import reglas_modernizacion_postcorreccion as reglas  # Only its detection of Latin words (_latin_mask)

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# Point corrections, book by book. Found among the 200 words with the highest TF-IDF of each book
# (scikit-learn TfidfVectorizer) that are not in the Spanish dictionary and are one letter away from a much more
# frequent word of the same book; every pair was reviewed by hand. Types: OCR errors ("tenesa" -> "teresa"),
# "ç" read as "g" or "c" ("fuergas" -> "fuerzas") and the author's spelling kept by the edition ("dibina" -> "divina").
# {book: {wrong form: right form}}:
corrections = {}
with open(CORRECTIONS_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        corrections.setdefault(row["libro"], {})[row["incorrecta"]] = row["correcta"]
print(sum(len(c) for c in corrections.values()), "corrections in", len(corrections), "books")

792 corrections in 124 books


In [3]:
# Celda 3
# Lexical modernization of the whole corpus: "ansí" -> "así", "agora" -> "ahora", "mesmo" -> "mismo".
# Source: the compact lexicon of IMPACT-es (Sánchez-Martínez et al., "An open diachronic corpus of historical Spanish",
# Language Resources and Evaluation, 2013; digitisation.eu, CC BY-SA 3.0 / GPL v3), whose historical forms are
# linked to a modern form. Only safe pairs were kept: single words found in the corpus, with one modern form, whose
# old form is not a current word ("fijo", "yerro") nor Latin ("tam", "quis"), of 4 letters or more, and reviewed by
# hand; a few frequent pairs that the dictionary filter left out were added by hand ("mesmo", "proprio", "aqueste"):
lexical = {}
with open(LEXICAL_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        lexical[row["antigua"]] = row["moderna"]
print(len(lexical), "lexical pairs")

# Old spellings, accents and forms left by 6-modernizacion and by the list above: "padesció" -> "padeció",
# "creatura" -> "criatura", "defunctos" -> "difuntos", "egercicio" -> "ejercicio", "habia" -> "había", "anque" -> "aunque",
# "deste" -> "de este". Built from the words of the modernized texts that are not in the Spanish dictionary (Hunspell
# es_ES), by rewriting each one with historical spelling changes (sc/c, ct/t, g/j, x/j, s/x, e/i, o/u, v/b, qu/cu, z/c, h,
# ll/rl, -des/-is, accents) until it matches a dictionary word of the corpus; every pair was reviewed by hand, and Latin,
# Italian, valid clitic forms ("díjole") and halves of words cut at the end of a line were left out. Plus pairs added by hand
# (tipo "léxico" and "contracción", and some old forms that are in the dictionary: "tenia", "logar", "dino").
# Then the accents of 19th-century printing ("á" -> "a", "ó" -> "o") and the old endings of the second person plural
# ("viésedes" -> "vieseis", "hecistes" -> "hicisteis"), each checked against the dictionary.
# Types: tilde, grafía, OCR, léxico, contracción, desinencia:
forms = {}
with open(FORMS_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        forms[row["antigua"]] = row["moderna"]
# Old forms that are also proper names are changed only in lowercase: "dina" -> "digna", but "Dina", daughter of Jacob:
NAMES = {"dina"}
print(len(forms), "pairs of old forms")

1225 lexical pairs
5609 pairs of old forms


In [4]:
# Celda 4
# A word keeps its case: "Ansí" -> "Así", "AGORA" -> "AHORA", "deste" -> "de este":
def same_case(old, new):
    if len(old) > 1 and old.isupper():
        return new.upper()
    if old[0].isupper():
        return new[0].upper() + new[1:]
    return new

# Changes the words of one line found in pairs ({lowercase old form: new form}); tags and the text inside
# <foreign> (Latin and Italian passages) are left alone, and with skip_latin also the unmarked Latin words found as in
# 6-modernizacion; capitalized words in names are left alone. Returns the new line and the list of (old, new):
TAG = re.compile(r"(<[^>]+>)")
WORD = re.compile(r"[^\W\d_]+")

def change_line(line, pairs, inside_foreign, skip_latin=False, names=()):
    changes = []
    def replace(match):
        new = pairs.get(match.group(0).lower())
        if new is None or (match.group(0)[0].isupper() and match.group(0).lower() in names):
            return match.group(0)
        new = same_case(match.group(0), new)
        changes.append((match.group(0), new))
        return new
    pieces = TAG.split(line)
    for k in range(len(pieces)):
        if pieces[k].startswith("<foreign"):
            inside_foreign = True
        elif pieces[k].startswith("</foreign"):
            inside_foreign = False
        elif not pieces[k].startswith("<") and not inside_foreign:
            if skip_latin:
                matches = list(WORD.finditer(pieces[k]))
                latin = reglas._latin_mask([m.group(0) for m in matches])
                out, end = [], 0
                for m, is_latin in zip(matches, latin):
                    out.append(pieces[k][end:m.start()])
                    out.append(m.group(0) if is_latin else replace(m))
                    end = m.end()
                out.append(pieces[k][end:])
                pieces[k] = "".join(out)
            else:
                pieces[k] = WORD.sub(replace, pieces[k])
    return "".join(pieces), changes, inside_foreign

In [5]:
# Celda 5
# Every book: first its point corrections, then the lexical modernization, then the old forms. Run it once after 6-modernizacion: a second
# run finds almost nothing left to change and would overwrite the register:
register = []
totals = Counter()
for file_name in sorted(os.listdir(MODERN_FOLDER)):
    book = file_name[:-4]
    path = os.path.join(MODERN_FOLDER, file_name)
    with open(path, encoding="utf-8") as f:
        lines = f.read().split("\n")
    for list_name, pairs in [("corrección puntual", corrections.get(book, {})), ("modernización léxica", lexical),
                             ("formas antiguas", forms)]:
        if not pairs:
            continue
        inside_foreign = False
        for k in range(len(lines)):
            lines[k], changes, inside_foreign = change_line(lines[k], pairs, inside_foreign, list_name == "formas antiguas",
                                                              NAMES if list_name == "formas antiguas" else ())
            for old, new in changes:
                register.append([book, k + 1, old, new, list_name])
                totals[list_name] += 1
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
print(dict(totals))

{'corrección puntual': 2535, 'modernización léxica': 54867, 'formas antiguas': 55847}


In [6]:
# Celda 6
# The register, one row per changed word:
with open(REGISTER_FILE, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["libro", "linea", "antes", "despues", "lista"])
    writer.writerows(register)
print(len(register), "changes in", REGISTER_FILE)
print(Counter((row[2].lower(), row[3].lower()) for row in register).most_common(15))

113249 changes in registro-correcciones.csv
[(('ansí', 'así'), 4731), (('habemos', 'hemos'), 3140), (('agora', 'ahora'), 3068), (('mesmo', 'mismo'), 2603), (('sant', 'san'), 2596), (('sancto', 'santo'), 2590), (('oir', 'oír'), 2348), (('sanctos', 'santos'), 1321), (('mesma', 'misma'), 1099), (('anque', 'aunque'), 916), (('proprio', 'propio'), 876), (('priesa', 'prisa'), 757), (('sancta', 'santa'), 756), (('propria', 'propia'), 755), (('envia', 'envía'), 630)]
